# ⚽ Boca Juniors 2026 - Análisis de Datos y Modelo Predictivo

## Objetivo

El objetivo de este proyecto es analizar el rendimiento de Boca Juniors durante la temporada 2026 utilizando Python y herramientas de análisis de datos.

Se realizará:

- Exploración de datos (EDA)
- Limpieza y transformación de datasets
- Análisis del rendimiento del equipo
- Análisis de jugadores
- Lesiones y mercado de pases
- Métricas avanzadas (xG, tiros, posesión)
- Modelo predictivo de resultados
- Validación del modelo

Tecnologías utilizadas:

- Python
- Pandas
- NumPy
- Matplotlib

In [22]:
# ==========================================
# LIBRERÍAS
# ==========================================
from pathlib import Path
import sys
from src.eda import auditar_datasets

# Agrega la carpeta raíz del proyecto al PATH
ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import cargar_datasets

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)


In [14]:
# ==========================================
# CARGA DE LOS DATASETS
# ==========================================

datos = cargar_datasets()

print("=" * 50)
print("DATASETS CARGADOS")
print("=" * 50)

print(f"\nCantidad de datasets: {len(datos)}\n")

for nombre in sorted(datos.keys()):
    print(f"✓ {nombre}")

DATASETS CARGADOS

Cantidad de datasets: 20

✓ assists_official_wikipedia
✓ cards_official_wikipedia
✓ coach_comparison
✓ competition_summary
✓ fbref_player_summary
✓ fixture_clausura
✓ goalkeepers_official_wikipedia
✓ goals_by_match
✓ goals_official_wikipedia
✓ injuries
✓ matches
✓ matches_shooting
✓ ohiggins_leg1_match_stats
✓ ohiggins_leg1_player_stats
✓ player_advanced_clausura
✓ squad
✓ standings_tabla_anual
✓ team_summary_apertura
✓ team_xg_home_away
✓ transfers


In [15]:
# ==========================================
# RESUMEN GENERAL
# ==========================================

resumen = pd.DataFrame({
    "Dataset": datos.keys(),
    "Filas": [df.shape[0] for df in datos.values()],
    "Columnas": [df.shape[1] for df in datos.values()]
})

resumen = resumen.sort_values("Filas", ascending=False)

resumen

,Dataset,Filas,Columnas
15,squad,37,18
4,fbref_player_summary,31,15
7,goals_by_match,31,10
16,standings_tabla_anual,30,10
10,matches,28,10
11,matches_shooting,23,13
17,team_summary_apertura,22,2
5,fixture_clausura,19,4
1,cards_official_wikipedia,18,7
13,ohiggins_leg1_player_stats,15,14


In [23]:
auditoria = auditar_datasets(datos)

auditoria

,Dataset,Filas,Columnas,Nulos,Duplicados,Memoria (KB)
15,squad,37,18,176,0,10.47
4,fbref_player_summary,31,15,36,0,6.74
7,goals_by_match,31,10,9,0,14.70
16,standings_tabla_anual,30,10,0,0,4.06
10,matches,28,10,15,0,11.60
11,matches_shooting,23,13,1,0,7.90
17,team_summary_apertura,22,2,0,0,2.72
5,fixture_clausura,19,4,0,0,4.66
1,cards_official_wikipedia,18,7,0,0,2.85
13,ohiggins_leg1_player_stats,15,14,0,0,3.10


In [24]:
print("="*60)
print("AUDITORÍA GENERAL")
print("="*60)

print(f"Datasets: {len(datos)}")

print(f"Total de registros: {auditoria['Filas'].sum()}")

print(f"Total de columnas: {auditoria['Columnas'].sum()}")

print(f"Total de valores nulos: {auditoria['Nulos'].sum()}")

print(f"Datasets con nulos: {(auditoria['Nulos']>0).sum()}")

print(f"Datasets con duplicados: {(auditoria['Duplicados']>0).sum()}")

AUDITORÍA GENERAL
Datasets: 20
Total de registros: 330
Total de columnas: 187
Total de valores nulos: 322
Datasets con nulos: 9
Datasets con duplicados: 0


In [16]:
matches = datos["matches"]

matches.head()

,date,competition,round,venue,opponent,goals_for,goals_against,result,points,coach
0,2026-01-25,Apertura,1,H,Deportivo Riestra,1.0,0.0,W,3.0,Claudio Ubeda
1,2026-01-28,Apertura,2,A,Estudiantes (LP),1.0,2.0,L,0.0,Claudio Ubeda
2,2026-02-01,Apertura,3,H,Newell's Old Boys,2.0,0.0,W,3.0,Claudio Ubeda
3,2026-02-08,Apertura,4,A,Velez Sarsfield,1.0,2.0,L,0.0,Claudio Ubeda
4,2026-02-15,Apertura,5,H,Platense,0.0,0.0,D,1.0,Claudio Ubeda


In [17]:
matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           28 non-null     object 
 1   competition    28 non-null     object 
 2   round          28 non-null     object 
 3   venue          28 non-null     object 
 4   opponent       28 non-null     object 
 5   goals_for      27 non-null     float64
 6   goals_against  27 non-null     float64
 7   result         27 non-null     object 
 8   points         16 non-null     float64
 9   coach          28 non-null     object 
dtypes: float64(3), object(7)
memory usage: 2.3+ KB


In [18]:
matches.describe(include="all")

,date,competition,round,venue,opponent,goals_for,goals_against,result,points,coach
count,28,28,28,28,28,27.000000,27.000000,27,16.000000,28
unique,28,6,28,3,24,NaN,NaN,3,NaN,2
top,2026-01-25,Apertura,1,H,Cruzeiro,NaN,NaN,W,NaN,Claudio Ubeda
freq,1,16,1,14,2,NaN,NaN,14,NaN,24
mean,NaN,NaN,NaN,NaN,NaN,1.333333,0.629630,NaN,1.875000,NaN
std,NaN,NaN,NaN,NaN,NaN,1.000000,0.791695,NaN,1.204159,NaN
min,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,0.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,NaN,1.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,NaN,2.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,2.000000,1.000000,NaN,3.000000,NaN


In [19]:
faltantes = matches.isnull().sum()

faltantes = faltantes[faltantes > 0]

faltantes.sort_values(ascending=False)

points           12
goals_for         1
goals_against     1
result            1
dtype: int64

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

plt.bar(
    auditoria["Dataset"],
    auditoria["Filas"]
)

plt.xticks(rotation=90)

plt.title("Cantidad de registros por dataset")

plt.ylabel("Filas")

plt.tight_layout()

plt.show()

NameError: name 'analizar_dataframe' is not defined